# Exercise 5 — OpsAgent

`OpsAgent` is the capstone class.  It wires together the `TaskStore`, `build_ops_tools`, `run_ops_loop`, and `OpsGuardrails` into one object with a simple `run(goal)` interface.  Every run — including blocked ones — is recorded in history for auditing.

In [ ]:
import json
from dataclasses import dataclass, field

def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    text = str(text)
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except Exception:
        return None
@dataclass
class OpsTask:
    id:          str
    title:       str
    description: str = ""
    status:      str = "pending"
    result:      str = ""

class TaskStore:
    def __init__(self):
        self._tasks = {}; self._counter = 0
    def add(self, title, description=""):
        self._counter += 1
        tid = f"task_{self._counter:03d}"
        t = OpsTask(id=tid, title=title, description=description)
        self._tasks[tid] = t; return t
    def get(self, task_id): return self._tasks.get(task_id)
    def all(self): return list(self._tasks.values())
    def pending(self): return [t for t in self._tasks.values() if t.status == "pending"]
    def update(self, task_id, status, result=""):
        t = self._tasks.get(task_id)
        if t: t.status = status
        if t and result: t.result = result
        return t
    def __len__(self): return len(self._tasks)
def build_ops_tools(store, executor_fn=None):
    def list_tasks(status=None):
        tasks = store.all() if status is None else [t for t in store.all() if t.status == status]
        if not tasks: return "No tasks found."
        return "\n".join("[" + t.id + "] [" + t.status + "] " + t.title for t in tasks)
    def run_task(task_id):
        t = store.get(task_id)
        if t is None: return "Error: task " + repr(task_id) + " not found"
        if t.status not in ("pending", "failed"): return "Task " + task_id + " is already " + t.status
        store.update(task_id, "running")
        try:
            result = str(executor_fn(t)) if executor_fn else "Completed: " + t.title
        except Exception as exc:
            store.update(task_id, "failed", str(exc)); return "Error: " + str(exc)
        store.update(task_id, "done", result)
        return "Task " + task_id + " done: " + result
    def check_status(task_id):
        t = store.get(task_id)
        if t is None: return "Error: task " + repr(task_id) + " not found"
        return "[" + task_id + "] " + t.title + ": " + t.status + (" — " + t.result if t.result else "")
    def generate_report(scope="all"):
        tasks = store.all()
        if not tasks: return "No tasks in store."
        counts = {}
        for t in tasks: counts[t.status] = counts.get(t.status, 0) + 1
        summary = ", ".join(str(v) + " " + k for k, v in sorted(counts.items()))
        return "Ops Report (" + str(scope) + "): " + str(len(tasks)) + " tasks — " + summary
    return {"list_tasks": list_tasks, "run_task": run_task,
            "check_status": check_status, "generate_report": generate_report}
def _validate_text(text, max_length=None, banned=None):
    text_str = str(text)
    if max_length is not None and len(text_str) > max_length:
        return False, "text exceeds max_length (" + str(len(text_str)) + " > " + str(max_length) + " chars)"
    if banned:
        lower = text_str.lower()
        for p in banned:
            if str(p).lower() in lower: return False, "banned pattern found: " + repr(p)
    return True, ""

class _Guard:
    def __init__(self, max_length=None, banned=None):
        self.max_length = max_length; self.banned = list(banned) if banned else []
    def check(self, text): return _validate_text(text, self.max_length, self.banned)

class _ApprovalGate:
    def __init__(self, approve_fn=None):
        self._fn = approve_fn if approve_fn is not None else (lambda a: True)
    def check(self, action):
        try: result = bool(self._fn(str(action)))
        except Exception: result = False
        return (True, "approved") if result else (False, "rejected by approval gate")

class _BudgetTracker:
    def __init__(self, max_calls=None):
        self.max_calls = max_calls; self._count = 0
    def ok(self):
        if self.max_calls is not None and self._count >= self.max_calls:
            return False, "budget exceeded (" + str(self._count) + "/" + str(self.max_calls) + " calls)"
        return True, ""
    def record(self): self._count += 1
    def reset(self): self._count = 0
    @property
    def count(self): return self._count

class OpsGuardrails:
    def __init__(self, max_budget=20, banned_inputs=None, approve_fn=None):
        self._input_guard = _Guard(max_length=500, banned=list(banned_inputs) if banned_inputs else [])
        self._budget = _BudgetTracker(max_calls=max_budget)
        self._gate   = _ApprovalGate(approve_fn=approve_fn)
    def check_input(self, query): return self._input_guard.check(str(query))
    def check_budget(self):
        ok, reason = self._budget.ok()
        if ok: self._budget.record()
        return ok, reason
    def check_approval(self, goal): return self._gate.check(str(goal))
    def reset(self): self._budget.reset()
    @property
    def budget_count(self): return self._budget.count
def _line_value(text, prefix):
    plow = prefix.lower()
    for line in str(text).splitlines():
        s = line.strip()
        if s.lower().startswith(plow): return s[len(prefix):].strip()
    return ""

def parse_ops_step(text):
    thought = _line_value(text, "Thought:")
    final   = _line_value(text, "Final Answer:")
    if final:
        return {"thought": thought, "action": None, "input": {}, "final": final}
    action = _line_value(text, "Action:")
    raw    = _line_value(text, "Input:")
    args   = safe_parse_json(raw) if raw else {}
    if not isinstance(args, dict): args = {}
    return {"thought": thought, "action": action or None, "input": args, "final": None}

def format_ops_step(step, observation=""):
    lines = []
    if step.get("thought"): lines.append("Thought: " + str(step["thought"]))
    if step.get("final"):   lines.append("Final Answer: " + str(step["final"]))
    elif step.get("action"):
        lines.append("Action: " + str(step["action"]))
        lines.append("Input: " + json.dumps(step.get("input", {})))
        if observation: lines.append("Observation: " + str(observation))
    return "\n".join(lines) + "\n"

def build_ops_prompt(goal, tools, scratchpad=""):
    tool_names = "\n".join("  - " + n for n in tools)
    system = "\n".join([
        "You are an autonomous ops agent. Complete the goal using the available tools.",
        "Available tools:", tool_names, "",
        "Reply with EXACTLY this format:",
        "Thought: <your reasoning>", "Action: <tool_name>", "Input: <json dict or {}>", "",
        "OR if the goal is fully accomplished:",
        "Thought: <your reasoning>", "Final Answer: <summary of what was accomplished>",
    ])
    user = "Goal: " + str(goal)
    if scratchpad: user += "\n\n" + scratchpad.rstrip()
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def run_ops_step(goal, tools, scratchpad="", llm_fn=None):
    return parse_ops_step(call_llm(build_ops_prompt(goal, tools, scratchpad), llm_fn=llm_fn))
def run_ops_loop(goal, tools, llm_fn=None, guardrails=None, max_iterations=10):
    scratchpad = ""; trace = []
    for iteration in range(max_iterations):
        if guardrails is not None:
            ok, reason = guardrails.check_budget()
            if not ok:
                return {"answer": "Stopped: " + reason, "trace": trace,
                        "iterations": iteration, "stopped": "budget"}
        step = run_ops_step(goal, tools, scratchpad, llm_fn=llm_fn)
        if step.get("final"):
            step["observation"] = ""; trace.append(step)
            return {"answer": step["final"], "trace": trace,
                    "iterations": iteration + 1, "stopped": ""}
        action = step.get("action") or ""
        args   = step.get("input") or {}
        obs = (str(tools[action](**args)) if action and action in tools
               else "Error: unknown tool " + repr(action))
        step["observation"] = obs; trace.append(step)
        scratchpad += format_ops_step(step, obs)
    return {"answer": "Task incomplete after max iterations.", "trace": trace,
            "iterations": max_iterations, "stopped": "max_iterations"}

# ── Exercise: implement OpsAgent ─────────────────────────────────────────────

class OpsAgent:
    """Autonomous ops agent with guardrails and full audit history."""

    def __init__(self, store=None, guardrails=None, llm_fn=None,
                 max_iterations=10):
        # TODO: store all args; self._history = []
        self._store = store or TaskStore()
        self._guardrails = guardrails
        self._llm_fn = llm_fn
        self.max_iterations = max_iterations
        self._history = []

    def run(self, goal, executor_fn=None):
        """Run the agent on a goal.

        Returns a record dict:
          {goal, answer, blocked, reason, trace, iterations, stopped}
        """
        # TODO:
        # 1. If guardrails: check_input(goal) → if not ok, return blocked record
        #    check_approval(goal) → if not approved, return blocked record
        # 2. tools = build_ops_tools(self._store, executor_fn=executor_fn)
        # 3. result = run_ops_loop(goal, tools, llm_fn=self._llm_fn,
        #             guardrails=self._guardrails, max_iterations=self.max_iterations)
        # 4. Build record dict, append to self._history, return it
        return {"goal": goal, "answer": None, "blocked": False, "reason": "",
                "trace": [], "iterations": 0, "stopped": ""}

    def history(self):
        # TODO: return list(self._history)
        return []

    def clear_history(self):
        # TODO: self._history.clear()
        pass


### Checks

In [ ]:
checks = 0

# Mock that immediately gives a final answer
_final_llm = lambda m: "Thought: done\nFinal Answer: All tasks processed."
_exec       = lambda t: "Simulated: " + t.title

# 1 — OpsAgent constructs and has expected attributes
try:
    agent = OpsAgent(llm_fn=_final_llm)
    assert hasattr(agent, "max_iterations") and hasattr(agent, "_store")
    checks += 1; print("✅ 1 OpsAgent constructs correctly")
except Exception as e:
    print("❌ 1:", e)

# 2 — run() executes a goal and returns a valid record
try:
    store = TaskStore(); store.add("Lint check"); store.add("Tests")
    agent = OpsAgent(store=store, llm_fn=_final_llm)
    r = agent.run("Run all tasks", executor_fn=_exec)
    assert r["answer"] == "All tasks processed."
    assert not r["blocked"] and r["goal"] == "Run all tasks"
    assert isinstance(r["trace"], list)
    checks += 1; print("✅ 2 run() executes goal and returns valid record")
except Exception as e:
    print("❌ 2:", e)

# 3 — guardrail blocks bad input; blocked record in history
try:
    g = OpsGuardrails(banned_inputs=["rm -rf"])
    agent = OpsAgent(guardrails=g, llm_fn=_final_llm)
    r = agent.run("please rm -rf /")
    assert r["blocked"] and "input" in r["reason"]
    assert r["answer"] is None
    checks += 1; print("✅ 3 guardrail blocks bad input; record stored in history")
except Exception as e:
    print("❌ 3:", e)

# 4 — run_ops_loop stops at max_iterations
try:
    never_done = lambda m: "Thought: still working\nAction: list_tasks\nInput: {}"
    store = TaskStore(); store.add("Task")
    agent = OpsAgent(store=store, llm_fn=never_done, max_iterations=3)
    r = agent.run("impossible goal", executor_fn=_exec)
    assert r["stopped"] == "max_iterations" and r["iterations"] == 3
    checks += 1; print("✅ 4 run_ops_loop stops at max_iterations")
except Exception as e:
    print("❌ 4:", e)

# 5 — history() grows with each run
try:
    agent = OpsAgent(llm_fn=_final_llm)
    agent.run("goal 1"); agent.run("goal 2"); agent.run("goal 3")
    assert len(agent.history()) == 3
    checks += 1; print("✅ 5 history() grows with each run()")
except Exception as e:
    print("❌ 5:", e)

# 6 — clear_history() empties history
try:
    agent = OpsAgent(llm_fn=_final_llm)
    agent.run("something")
    agent.clear_history()
    assert agent.history() == []
    checks += 1; print("✅ 6 clear_history() empties history")
except Exception as e:
    print("❌ 6:", e)

print(f"\n{checks}/6 checks passed!")
